# Object Detection

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/tnview-website/blob/main/pages/geoai/09-object-detection.ipynb)

## Introduction

Image recognition assigns a single label to an entire image tile, while semantic segmentation labels every pixel but cannot distinguish individual objects. Many geospatial applications -- such as counting vehicles, locating ships, or inventorying solar panels -- require identifying and localizing each object separately.

Object detection addresses this by producing a set of bounding boxes, each with a class label and confidence score, that localize individual objects within an image. In geospatial contexts, these bounding boxes can be georeferenced and converted to vector features for GIS workflows.

This tutorial covers the foundations of object detection for geospatial imagery, including key concepts, major architectural families, and a hands-on workflow for training and evaluating a Faster R-CNN v2 detector on the [NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip) dataset using the `geoai` package.

## Learning Objectives

By the end of this tutorial, you will be able to:

- Explain how object detection differs from image classification and semantic segmentation
- Describe bounding boxes, confidence scores, Non-Maximum Suppression, and anchor boxes
- Compare two-stage, single-stage, transformer-based, and zero-shot detection architectures
- Prepare detection datasets in COCO annotation format
- Train a multi-class object detection model using the `geoai` package
- Evaluate detection performance using COCO-style mean Average Precision (mAP)
- Run inference on new imagery and visualize detection results
- Publish trained models to Hugging Face Hub and run inference from hosted models

## Understanding Object Detection

### Classification vs. Detection

Image classification assigns one label per image (e.g., "residential area"), but says nothing about where specific objects are or how many exist. Object detection outputs a variable number of bounding boxes, each with a class label and confidence score, enabling counting and localization of individual objects.

Semantic segmentation provides pixel-level labels but does not separate individual objects. Object detection provides object-level localization but does not delineate exact boundaries. Instance segmentation combines both capabilities.

### Key Concepts

**Bounding boxes** are the fundamental output of a detection model. Each box is defined by four coordinates specifying a rectangle around a detected object, commonly in corner format (x_min, y_min, x_max, y_max) or center format (center_x, center_y, width, height). In geospatial applications, pixel coordinates are transformed into geographic coordinates using the image's affine transform.

**Confidence scores** range from 0 to 1 and indicate the model's certainty about each detection. A high confidence threshold reduces false positives but may miss objects, while a low threshold captures more objects but introduces more false detections.

**Intersection over Union (IoU)** measures overlap between a predicted box and a ground truth box as the area of intersection divided by the area of union. It is used during both training and evaluation to determine whether a detection is correct.

**Non-Maximum Suppression (NMS)** removes redundant overlapping detections by keeping the highest-confidence box and suppressing others that exceed an IoU threshold. This is essential for clean outputs in dense scenes.

**Anchor boxes** are pre-defined bounding box templates at various scales and aspect ratios placed across the image. Models predict offsets relative to these anchors, simplifying learning compared to predicting absolute coordinates.

## Detection Architectures

### Two-Stage Detectors

Two-stage detectors split detection into proposal generation and classification. **Faster R-CNN** first uses a Region Proposal Network (RPN) to identify candidate regions, then classifies and refines each proposal through separate heads.

Two-stage detectors achieve high accuracy and handle multiple scales well, making them suitable for geospatial imagery. However, their sequential design makes them slower than single-stage alternatives. This tutorial uses Faster R-CNN v2 with a ResNet-50 FPN backbone.

### Single-Stage Detectors

Single-stage detectors predict bounding boxes and class labels directly from the feature map in one pass, making them significantly faster than two-stage approaches.

- **YOLO** (You Only Look Once) frames detection as dense prediction over a grid, with modern variants using multi-scale feature pyramids.
- **SSD** (Single Shot MultiBox Detector) predicts from multiple feature maps at different resolutions to handle varying object sizes.
- **RetinaNet** uses focal loss to address class imbalance by focusing training on hard examples. Supported in `geoai` as `retinanet_resnet50_fpn_v2`.
- **FCOS** is an anchor-free detector that predicts boxes directly at each spatial location. Supported in `geoai` as `fcos_resnet50_fpn`.

Single-stage detectors are preferred for real-time applications and processing large volumes of imagery.

### Transformer-Based Detectors

[DETR](https://huggingface.co/docs/transformers/model_doc/detr) (DEtection TRansformer) frames detection as a set prediction problem using a transformer encoder-decoder architecture. It eliminates anchors and NMS by using learned object queries and Hungarian matching during training.

DETR excels at global context reasoning but converges slowly and struggles with very small objects. Variants like Deformable DETR, DINO, and RT-DETR address these limitations.

### Zero-Shot Detection

Zero-shot detectors identify objects described by text prompts without task-specific training data. [OWL-ViT](https://huggingface.co/docs/transformers/en/model_doc/owlvit) combines a vision transformer with a text encoder, enabling detection via natural language queries like "solar panel" or "swimming pool."

[Grounding DINO](https://huggingface.co/docs/transformers/en/model_doc/grounding-dino) extends this by combining DINO-based detection with grounded language understanding, achieving strong zero-shot performance. These models are invaluable for rapid prototyping when labeled training data is unavailable.

### Choosing an Architecture

The choice depends on your application requirements:

- **Faster R-CNN** is a strong default when accuracy is the priority. It is the default in the `geoai` package.
- **YOLO** is preferred when processing speed matters, such as scanning large archives or near-real-time monitoring.
- **DETR and variants** suit scenes requiring global context reasoning and avoid anchor tuning.
- **Zero-shot detectors** (OWL-ViT, Grounding DINO) are ideal for exploratory analysis or when labeled data is unavailable.

The `geoai` package supports multiple detection architectures through the `model_name` parameter:

| Model Name                          | Type         | Notes                                  |
| ----------------------------------- | ------------ | -------------------------------------- |
| `fasterrcnn_resnet50_fpn_v2`        | Two-stage    | Default, good accuracy/speed trade-off |
| `fasterrcnn_mobilenet_v3_large_fpn` | Two-stage    | Fastest two-stage option               |
| `retinanet_resnet50_fpn_v2`         | Single-stage | Fast, handles class imbalance well     |
| `fcos_resnet50_fpn`                 | Single-stage | Anchor-free                            |
| `maskrcnn_resnet50_fpn`             | Two-stage    | Also produces instance masks (slowest) |

Match the detector's strengths to the scale of your target objects relative to image resolution. Objects spanning only a few pixels require architectures with strong small-object capabilities (e.g., Feature Pyramid Networks), while large objects are detected adequately by most architectures.

## Preparing Detection Datasets

### Annotation Formats

**COCO format** stores all annotations in a single JSON file with image metadata, category definitions, and bounding boxes as (x, y, width, height) in absolute pixel coordinates. This is the format used by the `geoai` package.

**YOLO format** uses one text file per image with normalized coordinates (`class_id center_x center_y width height`), making it lightweight and easy to script.

### The NWPU-VHR-10 Dataset

The [NWPU-VHR-10](https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip) dataset is a benchmark for object detection in very-high-resolution remote sensing imagery, containing 650 annotated images and 150 background-only images across 10 classes: airplane, ship, storage tank, baseball diamond, tennis court, basketball court, ground track field, harbor, bridge, and vehicle.

The `geoai` package provides `prepare_nwpu_vhr10` to automatically split annotated images into training and validation sets with COCO-format annotations.

## Evaluating Detection Results

### Mean Average Precision (mAP)

A detection is a **true positive** if its IoU with a ground truth box exceeds a threshold and it predicts the correct class; otherwise it is a **false positive**. Unmatched ground truth boxes are **false negatives**.

**Average Precision (AP)** for a single class is the area under the precision-recall curve. **Mean Average Precision (mAP)** averages AP across all classes.

### Precision-Recall Curves

Precision-recall curves plot precision vs. recall as the confidence threshold varies. A strong detector maintains high precision at high recall. Per-class curves reveal where a model struggles, guiding data collection and augmentation decisions.

### IoU Thresholds

- **mAP@0.5** requires 50% overlap (traditional PASCAL VOC metric, common in geospatial applications).
- **mAP@0.75** requires 75% overlap, rewarding precise localization.
- **mAP@0.5:0.95** averages across thresholds from 0.5 to 0.95 in steps of 0.05 (primary COCO metric).

The appropriate metric depends on your use case: mAP@0.5 may suffice for object counting, while stricter thresholds suit applications requiring tight localization.

## Installation

Uncomment the following line to install the required package.


In [ ]:
# %pip install geoai-py

## Import Libraries

The `geoai` package provides functions for the full object detection pipeline:

- `download_file`: downloads and extracts dataset archives
- `NWPU_VHR10_CLASSES`: list of the 10 object class names plus background
- `prepare_nwpu_vhr10`: splits the positive images into train/validation sets with COCO annotations
- `visualize_coco_annotations`: displays sample images with annotated bounding boxes
- `train_multiclass_detector`: trains a Faster R-CNN or other detection model
- `plot_detection_training_history`: plots loss curves over training epochs
- `evaluate_multiclass_detector`: computes COCO-style mAP metrics
- `multiclass_detection`: runs inference on a single image
- `visualize_multiclass_detections`: displays detection results on an image
- `batch_multiclass_detection`: runs inference on multiple images
- `push_detector_to_hub`: uploads a trained model to Hugging Face Hub
- `predict_detector_from_hub`: runs inference using a Hub-hosted model

In [ ]:
import os
import json

import geoai

## Download the NWPU-VHR-10 Dataset

Download and extract the dataset from Source Cooperative.

In [ ]:
url = "https://data.source.coop/opengeos/geoai/NWPU-VHR-10.zip"
data_dir = geoai.download_file(url)

In [ ]:
print(f"Dataset directory: {data_dir}")
print(f"Contents: {os.listdir(data_dir)}")

## Explore the Dataset

The dataset contains 10 object classes plus a background class at index 0.

In [ ]:
print(f"\nNWPU-VHR-10 Classes:")
for i, name in enumerate(geoai.NWPU_VHR10_CLASSES):
    print(f"  {i}: {name}")

```text
NWPU-VHR-10 Classes:
  0: background
  1: airplane
  2: ship
  3: storage_tank
  4: baseball_diamond
  5: tennis_court
  6: basketball_court
  7: ground_track_field
  8: harbor
  9: bridge
  10: vehicle
```

## Prepare the Dataset

Split the positive images into training and validation sets with COCO-format annotations.

In [ ]:
splits = geoai.prepare_nwpu_vhr10(data_dir, val_split=0.2, seed=42)

In [ ]:
print(f"Images directory: {splits['images_dir']}")
print(f"Number of classes: {splits['num_classes']}")
print(f"Class names: {splits['class_names']}")
print(f"Training images: {len(splits['train_image_ids'])}")
print(f"Validation images: {len(splits['val_image_ids'])}")

```text
Images directory: ./NWPU-VHR-10/positive_image_set
Number of classes: 11
Class names: ['background', 'airplane', 'ship', 'storage_tank', 'baseball_diamond', 'tennis_court', 'basketball_court', 'ground_track_field', 'harbor', 'bridge', 'vehicle']
Training images: 509
Validation images: 128
```

## Visualize Sample Annotations

Visualize sample images with ground truth bounding boxes to verify annotations before training.

In [ ]:
geoai.visualize_coco_annotations(
    annotations_path=splits["annotations_path"],
    images_dir=splits["images_dir"],
    num_samples=6,
    random=True,
    seed=1,
    cols=3,
    figsize=(12, 6),
)

## Train a Multi-Class Detection Model

The `train_multiclass_detector` function handles the full training pipeline. Key parameters:

- `model_name`: detection architecture (default `fasterrcnn_resnet50_fpn_v2`). Other options: `fasterrcnn_mobilenet_v3_large_fpn`, `retinanet_resnet50_fpn_v2`, `fcos_resnet50_fpn`, `maskrcnn_resnet50_fpn`.
- `class_names`: list of class names including background at index 0.
- `pretrained=True`: initializes the backbone with ImageNet weights for transfer learning.
- `num_epochs`: number of passes through the training set.
- `batch_size` and `learning_rate`: standard optimization hyperparameters.
- `val_split`: fraction of training split held out for internal validation.

In [ ]:
output_dir = "nwpu_output"

model_path = geoai.train_multiclass_detector(
    images_dir=splits["images_dir"],
    annotations_path=splits["train_annotations"],
    output_dir=output_dir,
    model_name="fasterrcnn_resnet50_fpn_v2",
    class_names=splits["class_names"],
    num_channels=3,
    batch_size=4,
    num_epochs=10,
    learning_rate=0.005,
    val_split=0.1,
    seed=42,
    pretrained=True,
    verbose=True,
)

## Plot Training Metrics

Plot training loss, validation IoU, and learning rate schedule over epochs.

In [ ]:
geoai.plot_detection_training_history(
    history_path=os.path.join(output_dir, "training_history.pth"),
)

## Evaluate with COCO Metrics

Run the trained model on the validation set and compute COCO-style mAP metrics across all 10 object classes.

In [ ]:
metrics = geoai.evaluate_multiclass_detector(
    model_path=model_path,
    images_dir=splits["images_dir"],
    annotations_path=splits["val_annotations"],
    num_classes=splits["num_classes"],
    class_names=splits["class_names"][1:],  # Exclude background
    batch_size=4,
)

```text
Evaluation Results:
  mAP@0.5:        0.7312
  mAP@0.75:       0.4936
  mAP@[0.5:0.95]: 0.4428

  Per-class AP@0.5:
    AP@0.5/airplane: 0.7106
    AP@0.5/baseball_diamond: 0.7885
    AP@0.5/basketball_court: 0.8957
    AP@0.5/bridge: 0.9052
    AP@0.5/ground_track_field: 0.7081
    AP@0.5/harbor: 0.5322
    AP@0.5/ship: 0.6349
    AP@0.5/storage_tank: 0.5624
    AP@0.5/tennis_court: 0.8967
    AP@0.5/vehicle: 0.6781
```

Large, distinctive objects like basketball courts and tennis courts achieve higher AP, while smaller or more ambiguous objects like vehicles, ships, and storage tanks score lower.

## Run Inference on Sample Images

Run inference on a validation image using `multiclass_detection`, which handles tiling, NMS, and result assembly.

In [ ]:
# Load validation data to pick a test image
with open(splits["val_annotations"], "r") as f:
    val_data = json.load(f)

test_img_info = val_data["images"][0]
test_img_path = os.path.join(splits["images_dir"], test_img_info["file_name"])
print(f"Test image: {test_img_path}")

In [ ]:
output_raster = "nwpu_detection_output.tif"

result_path, inference_time, detections = geoai.multiclass_detection(
    input_path=test_img_path,
    output_path=output_raster,
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
    num_channels=3,
)

print(f"\nInference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

```text
Collected 27 detections before NMS
After NMS: 7 detections
Multi-class detection completed in 0.14 seconds
Final detections: 7
  harbor: 6 detections
  bridge: 1 detections
Saved multi-class detection to nwpu_detection_output.tif

Inference time: 0.14s
Total detections: 7
```

## Visualize Detections

Overlay detected bounding boxes on the image, colored by class and labeled with confidence scores.

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=test_img_path,
    detections=detections,
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## Batch Inference on Multiple Images

Process multiple images at once and visualize results in a grid.

In [ ]:
val_image_paths = [
    os.path.join(splits["images_dir"], img["file_name"])
    for img in val_data["images"][:4]
]

results = geoai.batch_multiclass_detection(
    image_paths=val_image_paths,
    output_dir="nwpu_batch_output",
    model_path=model_path,
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
    confidence_threshold=0.5,
    num_channels=3,
    figsize=(16, 12),
)

Predictions are not perfect -- you may notice false positives (e.g., a baseball diamond predicted over water) and false negatives (e.g., a missed ship).

## Publish and Reuse Models

### Push to Hugging Face Hub

Upload your trained model to Hugging Face Hub for sharing. You need a free account and a write-access token.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
url = geoai.push_detector_to_hub(
    model_path=model_path,
    repo_id="your-username/nwpu-vhr10-fasterrcnn",
    model_name="fasterrcnn_resnet50_fpn_v2",
    num_classes=splits["num_classes"],
    class_names=splits["class_names"],
)

### Run Inference from Hub

Download a model from the Hub and run inference without any local checkpoint.

In [ ]:
sample_img_path = os.path.join(splits["images_dir"], "608.jpg")

result_path, inference_time, detections = geoai.predict_detector_from_hub(
    input_path=sample_img_path,
    output_path="hub_detection.tif",
    repo_id="giswqs/nwpu-vhr10-fasterrcnn",
    confidence_threshold=0.5,
)

print(f"Inference time: {inference_time:.2f}s")
print(f"Total detections: {len(detections)}")

# Clean up
if os.path.exists("hub_detection.tif"):
    os.remove("hub_detection.tif")

```text
Collected 34 detections before NMS
After NMS: 8 detections
Multi-class detection completed in 0.13 seconds
Final detections: 8
  baseball_diamond: 1 detections
  tennis_court: 4 detections
  basketball_court: 3 detections
Saved multi-class detection to hub_detection.tif
Inference time: 0.13s
Total detections: 8
```

In [ ]:
geoai.visualize_multiclass_detections(
    image_path=sample_img_path,
    detections=detections,
    class_names=geoai.NWPU_VHR10_CLASSES,
    confidence_threshold=0.5,
    figsize=(12, 10),
)

## Key Takeaways

1. Object detection localizes and classifies individual objects with bounding boxes, filling the gap between image classification and semantic segmentation.

2. Bounding boxes, IoU, NMS, and anchor boxes are foundational concepts for how detection models generate, refine, and filter predictions.

3. Two-stage detectors (Faster R-CNN) prioritize accuracy, single-stage detectors (YOLO, RetinaNet, FCOS) prioritize speed, transformer-based detectors (DETR) simplify the pipeline, and zero-shot detectors (OWL-ViT) eliminate the need for training data.

4. The NWPU-VHR-10 dataset provides a 10-class benchmark for multi-class detection in remote sensing with COCO-format annotations.

5. The `geoai` package streamlines the full detection workflow from dataset preparation through training, evaluation, and inference.

6. COCO-style mAP metrics at multiple IoU thresholds provide comprehensive evaluation, with per-class AP revealing strengths and weaknesses.

7. Confidence threshold tuning balances precision and recall depending on whether false positives or false negatives are more costly.

8. Publishing models to Hugging Face Hub enables sharing and reuse without local training.